# Task 47: Versioned Model Export

Train a scratch logistic model, package its explicit state and preprocessing metadata, export with compressed joblib settings, reload, validate compatibility, and compare predictions.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import joblib

candidates=[Path('heart.csv'),Path('Task_47_Versioned_Model_Export/heart.csv'),Path('Heart_Disease_Tasks/Task_47_Versioned_Model_Export/heart.csv')]
csv_path=next(path for path in candidates if path.is_file())
data=pd.read_csv(csv_path).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[features].to_numpy(float),data.target.to_numpy(int)

def stratified_split(labels,fraction,seed):
    rng=np.random.default_rng(seed);first=[];second=[]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);rng.shuffle(ids);cut=round(fraction*len(ids));first.extend(ids[:cut]);second.extend(ids[cut:])
    rng.shuffle(first);rng.shuffle(second);return np.array(first),np.array(second)

trainval,test=stratified_split(y,.8,47)
train_local,val_local=stratified_split(y[trainval],.8,48)
train,val=trainval[train_local],trainval[val_local]
X_train,X_val,y_train,y_val=X[train],X[val],y[train],y[val]
train_mean,train_scale=X_train.mean(0),X_train.std(0);train_scale[train_scale==0]=1
X_train_scaled=(X_train-train_mean)/train_scale;X_val_scaled=(X_val-train_mean)/train_scale

class ManualLogistic:
    def __init__(self,learning_rate=.06,iterations=1400,l2=.02):self.learning_rate,self.iterations,self.l2=learning_rate,iterations,l2
    def fit(self,x,labels):
        self.weights=np.zeros(x.shape[1]);self.bias=0.
        for _ in range(self.iterations):
            probability=1/(1+np.exp(-np.clip(x@self.weights+self.bias,-35,35)));error=probability-labels
            self.weights-=self.learning_rate*(x.T@error/len(labels)+self.l2*self.weights);self.bias-=self.learning_rate*error.mean()
        return self
    def predict_proba(self,x):return 1/(1+np.exp(-np.clip(x@self.weights+self.bias,-35,35)))
    def predict(self,x,threshold=.5):return (self.predict_proba(x)>=threshold).astype(int)


def classification_metrics(actual,predicted):
    tp=np.sum((actual==1)&(predicted==1));tn=np.sum((actual==0)&(predicted==0));fp=np.sum((actual==0)&(predicted==1));fn=np.sum((actual==1)&(predicted==0))
    precision=tp/(tp+fp) if tp+fp else 0.;recall=tp/(tp+fn) if tp+fn else 0.
    return {'accuracy':float((tp+tn)/len(actual)),'precision':float(precision),'recall':float(recall),'f1':float(2*precision*recall/(precision+recall)) if precision+recall else 0.}

parameters={'learning_rate':.06,'iterations':1400,'l2':.02}
selection_model=ManualLogistic(**parameters).fit(X_train_scaled,y_train)
print('Validation metrics:',classification_metrics(y_val,selection_model.predict(X_val_scaled)))

# Refit selected settings on the full development split; keep test rows unseen until final assessment.
X_dev,X_test=X[trainval],X[test];y_dev,y_test=y[trainval],y[test]
means,scales=X_dev.mean(0),X_dev.std(0);scales[scales==0]=1
X_dev_scaled=(X_dev-means)/scales;X_test_scaled=(X_test-means)/scales
model=ManualLogistic(**parameters).fit(X_dev_scaled,y_dev)
test_metrics=classification_metrics(y_test,model.predict(X_test_scaled))

# Store a versioned state dictionary rather than serializing a live estimator object.
artifact={
    'schema_version':1,'model_version':'1.0.0','created_utc':datetime.now(timezone.utc).isoformat(),
    'model_class':'ManualLogistic','parameters':parameters,
    'state':{'weights':model.weights.tolist(),'bias':float(model.bias)},
    'feature_names':features,'feature_count':len(features),'target_name':'target','threshold':.5,
    'preprocessing':{'kind':'standardize','means':means.tolist(),'scales':scales.tolist()},
    'dataset_sha256':hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    'runtime':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'joblib':joblib.__version__}
}
artifact_dir=Path('artifacts');artifact_dir.mkdir(exist_ok=True)
artifact_path=artifact_dir/'heart_logistic_v1.0.0.joblib'
joblib.dump(artifact,artifact_path,compress=('zlib',3),protocol=5)

# Validate the schema, restore model state, and check round-trip prediction parity.
loaded=joblib.load(artifact_path)
assert loaded['schema_version']==1 and loaded['model_class']=='ManualLogistic'
restored=ManualLogistic(**loaded['parameters'])
restored.weights=np.asarray(loaded['state']['weights'],dtype=float);restored.bias=float(loaded['state']['bias'])
assert restored.weights.shape==(loaded['feature_count'],)
restored_predictions=restored.predict(X_test_scaled,loaded['threshold'])
assert np.array_equal(restored_predictions,model.predict(X_test_scaled,.5))
digest=hashlib.sha256(artifact_path.read_bytes()).hexdigest()
print('Model version:',loaded['model_version'],'schema:',loaded['schema_version'])
print('Artifact:',artifact_path,'bytes:',artifact_path.stat().st_size,'SHA-256:',digest)
print('Reload prediction parity: passed')
print('Held-out test metrics:',test_metrics)
print('Only load joblib artifacts from trusted sources; keep the recorded runtime and feature order with deployment code.')

Validation metrics: {'accuracy': 0.75, 'precision': 0.7692307692307693, 'recall': 0.7692307692307693, 'f1': 0.7692307692307693}
Model version: 1.0.0 schema: 1
Artifact: artifacts/heart_logistic_v1.0.0.joblib bytes: 895 SHA-256: 0f8acc758c0db42f9da58941da900e311ccdabbbeb3d882af1024a0cfdd20eb3
Reload prediction parity: passed
Held-out test metrics: {'accuracy': 0.8032786885245902, 'precision': 0.7560975609756098, 'recall': 0.9393939393939394, 'f1': 0.8378378378378378}
Only load joblib artifacts from trusted sources; keep the recorded runtime and feature order with deployment code.
